In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'

In [2]:
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using header file: ../oauth.json
Test Passed in 3.37 seconds
Using ytmusicapi version: 1.3.1
Loaded 462 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [3]:
playlist_names = [
]
MIN_RADIO_LIKE_TO_SPLIT=2
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_radio_playlist(pl_info, 
        move_like=True, create_like_playlist=True, remove_dislike=True, remove_not_like=True,
        min_num_like=MIN_RADIO_LIKE_TO_SPLIT,  sleep=1, verbose=True,
    )


****************************************************************************************************
{'name': 'Shoegaze Nu radio', 'removed_dislike': 2, 'moved_like': 24, 'removed_not_like': 0, 'like_and_not_like': 0}
Moved 24 LIKE entries from Shoegaze Nu radio to PLWptjpDqazOwCkXr-SJX9uW142Ova6Cq_
Removed 2 DISLIKE entries from Shoegaze Nu radio


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [5]:
playlist_names = [

]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=USE_CACHE)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


Loaded 266580 playounts from 104345 tracks
Created sorted pl: PLWptjpDqazOyKq8ywE6hYflYP6dtCJd1g, and  deleted original pl: PLWptjpDqazOy4zp29oWVHfXetof6lb34q
Created sorted pl: PLWptjpDqazOwWxRXEca6d0dfTj3qVtXWs, and  deleted original pl: PLWptjpDqazOxdeuX4BqrgPyFe8HC-aD0r


## Save playlist backup tsv

In [6]:
PLAYLIST_NAME = 'zz not like 2'
USE_CACHE = False
_pl_id = Y.query_by_title(PLAYLIST_NAME).playlistId
_pl_tracks, _pl_metadata = Y.save_playlist_tsv(Y.playlist_get_info(_pl_id, use_cache=USE_CACHE))

                                                                  0
id                               PLWptjpDqazOymmTOH-SrVAbtGxYiosu73
privacy                                                     PRIVATE
title                                                 zz not like 2
description                                                    None
author            {'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFUM...
views                                                          None
duration                                                   7+ hours
trackCount                                                     1496
related                                                          []
duration_seconds                                             354872


## Update _not_like tsv

In [4]:
not_like_tracks = Y.collect_all_not_like_tracks_from_tsvs()
not_like_tracks.to_csv(Y.not_like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to not_like tsv: {Y.not_like_tsv}')

Found 2 not like playlists out of the 476 total
Updated not liked tracks, contains 4996 entries.
Saved 4996 entries to not_like tsv: ../playlists/_not_liked_tracks.tsv


## Update _like tsv and print like coverage

In [12]:
like_tracks = Y.collect_all_like_tracks_from_tsvs()
like_tracks.to_csv(Y.like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to like tsv: {Y.like_tsv}')

Found 112 like playlists out of the 474 total
Beats indie Chill.tsv	98.8% currently liked (of 83 total tracks) 
Bossa Nova.tsv	95.2% currently liked (of 21 total tracks) 
Brass n chill.tsv	96.5% currently liked (of 258 total tracks) 
Chillwave.tsv	98.1% currently liked (of 367 total tracks) 
Folk.tsv	99.2% currently liked (of 249 total tracks) 
Grunge.tsv	100.0% currently liked (of 79 total tracks) 
Hip Hop 1990s.tsv	98.7% currently liked (of 620 total tracks) 
Hip Hop 2000s.tsv	100.0% currently liked (of 92 total tracks) 
Hip Hop Hits.tsv	99.1% currently liked (of 225 total tracks) 
Hip hop It Was a Good Day.tsv	98.7% currently liked (of 77 total tracks) 
Indie 1990s Rock.tsv	100.0% currently liked (of 76 total tracks) 
Indie 2000s.tsv	95.7% currently liked (of 233 total tracks) 
Instrumentals Acoustic like.tsv	100.0% currently liked (of 11 total tracks) 
Jazz Guitar.tsv	98.9% currently liked (of 93 total tracks) 
Oldies.tsv	99.1% currently liked (of 1158 total tracks) 
Post-Punk 1970

## Get Public Playlists

In [13]:
Y.get_playlists_by_privacy(privacy='PUBLIC')
# last run 6-2023, 3 public playlists

Found public playlist named: Liked Music
Found public playlist named: indie loose
Found public playlist named: Chill Supermix
Found public playlist named: Episodes for Later


title                                                Liked Music
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
author                                                       NaN
title                                                indie loose
playlistId                    PLWptjpDqazOygyoaRpT447v7SENKpPNfi
thumbnails     [{'url': 'https://yt3.googleusercontent.com/tZ...
description                                Jake G • 1,499 tracks
count                                                      1,499
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
title                                             Chill Supermix
playlistId           RDTMAK5uy_nzfwl2UYv7htL7wDoxbX8Pp6UAFBd92cQ
thumbnails     [{'url': 'https://music.youtube.com/image/mixa...
description              

## Get Playlist Counts

In [ ]:
%%time
playlist_file = os.path.join(PLAYLIST_TSV_DIR, '_playlist_radio_counts.tsv')
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
radio_counts_df.to_csv(playlist_file, sep='\t', index=False)
# Note: now part of ytmusic backup

radio_counts_df.head(20)
# last run 10-29-2023


CPU times: user 26.1 s, sys: 1.12 s, total: 27.2 s
Wall time: 4min 21s


,title,track_count,duration_hours,privacy,playlist_id
105,ambient modern radio,36,4,PRIVATE,PLWptjpDqazOwO_d5ayc93-4pTuls5NCxG
133,x_r.MusicToSleepTo_tracks_radio,40,6,PRIVATE,PLWptjpDqazOzfumkZJ8sZi_fKVxXLZOfI
3,jazz blue note sampled radio,42,5,UNLISTED,PLWptjpDqazOxdeuX4BqrgPyFe8HC-aD0r
132,x_r.chicagohouse_tracks_radio,47,5,PRIVATE,PLWptjpDqazOzwOd5TBZvCFqkwH5hcuMbu
121,Hip hop It Was a Good Day radio,47,3,PRIVATE,PLWptjpDqazOz-tIqgqECFiRHLqE5IYtdz
124,jazz for spies radio,53,3,PRIVATE,PLWptjpDqazOyj2ARv0edtYtxjiscELzT3
19,x_r.dancepunk_tracks_radio,55,4,PRIVATE,PLWptjpDqazOxljQ9-i_YmrvGnryDPTegV
129,Produced by DJ Premier radio,55,4,PRIVATE,PLWptjpDqazOxsFhbAvcJaTI7CmK4vQgtU
123,Instrumentals Acoustic radio,55,3,PRIVATE,PLWptjpDqazOwMXA073PbBvjxkW0VTEC05
20,christmas crooners radio,58,3,PRIVATE,PLWptjpDqazOx6WdsiNzxLHnAdh8v1IjTa


# One Off

## Upload playlist from tsv backup

In [10]:
PLAYLIST_NAME = 'hiphop 80s'
playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)


Generating hiphop 80s ytmusic playlist for 25 tracks
Saved 25 hiphop 80s tracks playlist with id: PLWptjpDqazOyLve3SIyfcKJg1OOCyGpuC


## Query

In [ ]:
Y.query_by_title(playlist_name)

title                                                 jazz radio
playlistId                    PLWptjpDqazOxgrXKM4xfJ-SBRahEDfXmG
thumbnails     [{'url': 'https://yt3.googleusercontent.com/zS...
description                                  Jake G • 797 tracks
count                                                        797
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
Name: 20, dtype: object

### Generate playlist froma a list of albums

In [ ]:
name = 'y_2022_albums_to_listen_to'
desc = 'manually selected albums to top off 2022 albums'
albums_to_add = [
    "SZA - SOS",
    "Perfume Genius - Ugly Season",
    "Arctic Monkeys - The Car",
    "The Beths - Expert in a Dying Field",
    "alt-J - The Dream",
     "Soichi Terada (寺田創一) - Asakusa Light",
    "Everything Everything - Raw Data Feel",
    "The Mountain Goats - Bleed Out",
    "Metric - Formentera",
    "Orville Peck - Bronco",
    "Hurray for the Riff Raff - Life on Earth",
    "Freddie Gibbs - $oul $old $eparately",
    "Tove Lo - Dirt Femme",
    "Oren Ambarchi - Shebang",
    "Hatchie - Giving the World Away",
    "Florence + the Machine - Dance Fever",
    "Richard Dawson - The Ruby Cord",
    "Charlotte Adigéry & Bolis Pupul - Topical Dancer",
    "Phoenix (FR) - Alpha Zulu",
    "Elder - Innate Passage",
    "The Black Angels - Wilderness of Mirrors",
    "Black Flower - Magma",
     "Bad Bunny - Un Verano Sin Ti",
     "Red Hot Chili Peppers - Return Of The Dream Canteen",

]

track_ids = []
for a in albums_to_add:
    match = {}
    res = Y.yt.search(query=a, filter='albums', limit=1)
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    if len(res) == 0 or res[0].get('browseId') == None:
        print(f'Skipping query: {a} bad result: {res}')
        continue
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    print(f'Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album}')

pl_id = Y.yt.create_playlist( title=name,  description=desc, video_ids=track_ids)
print(f'Generated playlist: {name} with id {pl_id}')